# einops-reduce composite — cx9: group-flatten then reduce within each group — avg-pool style

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `einops-rearrange-flatten`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "einops-rearrange-flatten"]
DD_SUBTOPICS = ["Einops: Reduce", "Einops: Rearrange-as-flatten"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's average-pool / patch-summary moves want to (a) RESHAPE so the pool window lives on a single named axis, then (b) REDUCE over that axis. `einops.rearrange` with parenthesised axis composition is the flatten step: `'b c (h1 h2) (w1 w2) -> b c (h1 w1) (h2 w2)'` regroups the spatial axes so every pool window is contiguous. Then `einops.reduce` over the inner `(h2 w2)` axis is the actual pooling op.

Composing them tests both atoms: you must (i) write a rearrange pattern that uses axis-composition via parens (atom: rearrange-flatten) and (ii) pick the right reduce aggregator + pattern (atom: reduce).

### Composite Exercise — group-flatten then reduce within each group — avg-pool style

**Atoms exercised together**: `einops-reduce`, `einops-rearrange-flatten`

Implement `cx9_pool_2x2_via_einops(x)` that takes a 4-D feature map of shape `(B, C, H, W)` with `H % 2 == 0` and `W % 2 == 0`, and returns a `(B, C, H//2, W//2)` average-pooled tensor.

Two atoms, one composition:

1. **Rearrange-flatten** — use `einops.rearrange(x, 'b c (h1 h2) (w1 w2) -> b c h1 w1 (h2 w2)', h2=2, w2=2)`. Note the trailing `(h2 w2)` — axis composition via parens collapses the 2×2 pool window onto a single inner axis of length 4. Bind h2/w2 by kwarg.
2. **Reduce** — call `einops.reduce(..., 'b c h1 w1 win -> b c h1 w1', 'mean')` to collapse the new `win` axis. Renaming `(h2 w2) -> win` happens in the rearrange step, so you may use any inner-axis name in the reduce.

Equivalent to `F.avg_pool2d(x, kernel_size=2)`, but the point is to do it with einops.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx9_pool_2x2_via_einops(x):
    raise NotImplementedError

def _test_cx9():
    import torch.nn.functional as F
    x = t.arange(1 * 2 * 4 * 6).reshape(1, 2, 4, 6).float()
    out = cx9_pool_2x2_via_einops(x)
    assert out.shape == (1, 2, 2, 3), f'expected (1,2,2,3), got {out.shape}'
    ref = F.avg_pool2d(x, kernel_size=2)
    assert t.allclose(out, ref), f'differs from avg_pool2d: {out} vs {ref}'

    # Case B: larger random batch.
    x2 = t.randn(3, 4, 8, 8)
    out2 = cx9_pool_2x2_via_einops(x2)
    assert out2.shape == (3, 4, 4, 4)
    assert t.allclose(out2, F.avg_pool2d(x2, kernel_size=2))

    # Case C: smallest legal — H=W=2 → output (B, C, 1, 1).
    x3 = t.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])
    out3 = cx9_pool_2x2_via_einops(x3)
    assert out3.shape == (1, 1, 1, 1)
    assert t.allclose(out3, t.tensor([[[[2.5]]]]))
    _dd_passed.add('cx9')

_test_cx9()

<details><summary>Show solution — cx9</summary>

```python
def cx9_pool_2x2_via_einops(x):
    # Atom A: rearrange-flatten — axis-composition via parens regroups the 2x2 pool window
    # onto a single trailing 'win' axis of length 4.
    regrouped = rearrange(
        x,
        'b c (h1 h2) (w1 w2) -> b c h1 w1 (h2 w2)',
        h2=2, w2=2,
    )
    # Atom B: reduce — collapse the 'win' axis with 'mean'.
    return reduce(regrouped, 'b c h1 w1 win -> b c h1 w1', 'mean')
```

The parenthesised pattern is the magic — `(h1 h2)` means 'this axis is the COMPOSITION of two logical axes h1 and h2'. einops needs `h2=2` as a kwarg to know how to factor the existing H into h1 × h2. Once the window lives on its own axis, reduce is just one more line.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx9',
        'subtopics': ["Einops: Reduce", "Einops: Rearrange-as-flatten"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()